# Inspection: generation of scene

## Imports

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import numpy as np
from robotblockset.tools import get_rbs_path, print_xml, print_xml_for_console

from robotblockset.transformations import rot_x, rot_y, rot_z, map_pose

import mujoco 
from robotblockset.mujoco.tools_mjcf import print_body_tree, actuators_for_joint

## Initialization

In [3]:
ROBOT = "ur10e"
GRIPPER = "robotiq_2f85"
MODEL_PATH = get_rbs_path() + "/mujoco/mjcf_models/"

## Generation of scene

### Basic scene

In [4]:
scene=mujoco.MjSpec.from_file(MODEL_PATH + "inspection_cell.xml")
scene.copy_during_attach = True
print_body_tree(scene.worldbody, scene)
# print_xml(scene.to_xml())

Body Tree for "world"
-Target
-main_table
-cell_enclosure
-small_table
-Odkovek
--
-O-0
--
-O-1
--
-O-2
--
-O-3
--
-camera1_stand
-camera2_stand


### Attach robot to table

Load robot specification

In [5]:
robot_spec = mujoco.MjSpec.from_file(MODEL_PATH + f"{ROBOT}.xml")
robot_spec.copy_during_attach = True
print_body_tree(robot_spec.worldbody, robot_spec)
robot_ctrl = np.array(robot_spec.keys[0].ctrl)
robot_qpos = np.array(robot_spec.keys[0].qpos)
# print_xml(robot_spec.to_xml())

Body Tree for "world"
-base
--base_link
---shoulder_link (Joints: shoulder_pan_joint-HINGE[Actuator: pos_shoulder_pan_joint])
----upper_arm_link (Joints: shoulder_lift_joint-HINGE[Actuator: pos_shoulder_lift_joint])
-----forearm_link (Joints: elbow_joint-HINGE[Actuator: pos_elbow_joint])
------wrist_1_link (Joints: wrist_1_joint-HINGE[Actuator: pos_wrist_1_joint])
-------wrist_2_link (Joints: wrist_2_joint-HINGE[Actuator: pos_wrist_2_joint])
--------wrist_3_link (Joints: wrist_3_joint-HINGE[Actuator: pos_wrist_3_joint])
---------flange


Attach robot to mounting plate and rename robot base

In [6]:
attachment_frame = scene.body("main_table").add_frame(pos=scene.site("ur10e_base_mount").pos, axisangle=[0, 0, 1, 0 * np.pi / 2])
attachment_frame.attach_body(robot_spec.body("base"), f"{ROBOT}_")
scene.body(f"{ROBOT}_base").name = ROBOT
print_body_tree(scene.worldbody, scene)

Body Tree for "world"
-Target
-main_table
--ur10e
---ur10e_base_link
----ur10e_shoulder_link (Joints: ur10e_shoulder_pan_joint-HINGE[Actuator: ur10e_pos_shoulder_pan_joint])
-----ur10e_upper_arm_link (Joints: ur10e_shoulder_lift_joint-HINGE[Actuator: ur10e_pos_shoulder_lift_joint])
------ur10e_forearm_link (Joints: ur10e_elbow_joint-HINGE[Actuator: ur10e_pos_elbow_joint])
-------ur10e_wrist_1_link (Joints: ur10e_wrist_1_joint-HINGE[Actuator: ur10e_pos_wrist_1_joint])
--------ur10e_wrist_2_link (Joints: ur10e_wrist_2_joint-HINGE[Actuator: ur10e_pos_wrist_2_joint])
---------ur10e_wrist_3_link (Joints: ur10e_wrist_3_joint-HINGE[Actuator: ur10e_pos_wrist_3_joint])
----------ur10e_flange
-cell_enclosure
-small_table
-Odkovek
--
-O-0
--
-O-1
--
-O-2
--
-O-3
--
-camera1_stand
-camera2_stand


Exclude contacts between robot base and table

In [7]:
scene.add_exclude(bodyname1=f"{ROBOT}_base_link", bodyname2="main_table")


### Attach gripper to robot

Load gripper specification

In [8]:
gripper_spec = mujoco.MjSpec.from_file(MODEL_PATH + f"{GRIPPER}.xml")
gripper_spec.copy_during_attach = True
print_body_tree(gripper_spec.worldbody, gripper_spec)
gripper_qpos= np.array(gripper_spec.keys[0].qpos)
gripper_ctrl= np.array(gripper_spec.keys[0].ctrl)


Body Tree for "world"
-hand
--left_driver (Joints: left_driver_joint-HINGE)
---left_coupler
--left_spring_link (Joints: left_spring_link_joint-HINGE)
---left_follower (Joints: left_follower-HINGE)
----left_pad
--right_driver (Joints: right_driver_joint-HINGE)
---right_coupler
--right_spring_link (Joints: right_spring_link_joint-HINGE)
---right_follower (Joints: right_follower_joint-HINGE)
----right_pad


Attach gripper to robot flange and redefine robot TCP

In [9]:
scene

In [10]:
attachment_frame = scene.body(f"{ROBOT}_flange").add_frame(pos=[0, 0, 0])
attachment_frame.attach_body(gripper_spec.body("hand"), f'{ROBOT}_gripper_')
scene.delete(scene.site(f"{ROBOT}_TCP"))
scene.site(f"{ROBOT}_gripper_hand_TCP").name = f"{ROBOT}_TCP"
print_body_tree(scene.worldbody, scene)

Body Tree for "world"
-Target
-main_table
--ur10e
---ur10e_base_link
----ur10e_shoulder_link (Joints: ur10e_shoulder_pan_joint-HINGE[Actuator: ur10e_pos_shoulder_pan_joint])
-----ur10e_upper_arm_link (Joints: ur10e_shoulder_lift_joint-HINGE[Actuator: ur10e_pos_shoulder_lift_joint])
------ur10e_forearm_link (Joints: ur10e_elbow_joint-HINGE[Actuator: ur10e_pos_elbow_joint])
-------ur10e_wrist_1_link (Joints: ur10e_wrist_1_joint-HINGE[Actuator: ur10e_pos_wrist_1_joint])
--------ur10e_wrist_2_link (Joints: ur10e_wrist_2_joint-HINGE[Actuator: ur10e_pos_wrist_2_joint])
---------ur10e_wrist_3_link (Joints: ur10e_wrist_3_joint-HINGE[Actuator: ur10e_pos_wrist_3_joint])
----------ur10e_flange
-----------ur10e_gripper_hand
------------ur10e_gripper_left_driver (Joints: ur10e_gripper_left_driver_joint-HINGE)
-------------ur10e_gripper_left_coupler
------------ur10e_gripper_left_spring_link (Joints: ur10e_gripper_left_spring_link_joint-HINGE)
-------------ur10e_gripper_left_follower (Joints: ur10e_

### Attach object to gripper

Object is alread defined in basic scene. Just attach it to the gripper

In [11]:
attachment_frame = scene.body(f"{ROBOT}_gripper_hand").add_frame(pos=scene.site(f"{ROBOT}_TCP").pos + [0, 0, 0.01], quat=map_pose(R=rot_z(np.pi, out="R") @ rot_y(np.pi, out="R") @ rot_z(np.pi/2, out="R"), out="Q"))
attachment_frame.attach_body(scene.body("Odkovek"), "Grasped_")
scene.delete(scene.body("Odkovek"))

In [12]:
0.313725 *255

79.99987499999999

It is a good practice to rename the actuator name for the gripper to equal the expected name for tha gripper actuator.

In [13]:
scene.actuators[-1].name = "robotiq_gripper"
print_body_tree(scene.worldbody, scene)

Body Tree for "world"
-Target
-main_table
--ur10e
---ur10e_base_link
----ur10e_shoulder_link (Joints: ur10e_shoulder_pan_joint-HINGE[Actuator: ur10e_pos_shoulder_pan_joint])
-----ur10e_upper_arm_link (Joints: ur10e_shoulder_lift_joint-HINGE[Actuator: ur10e_pos_shoulder_lift_joint])
------ur10e_forearm_link (Joints: ur10e_elbow_joint-HINGE[Actuator: ur10e_pos_elbow_joint])
-------ur10e_wrist_1_link (Joints: ur10e_wrist_1_joint-HINGE[Actuator: ur10e_pos_wrist_1_joint])
--------ur10e_wrist_2_link (Joints: ur10e_wrist_2_joint-HINGE[Actuator: ur10e_pos_wrist_2_joint])
---------ur10e_wrist_3_link (Joints: ur10e_wrist_3_joint-HINGE[Actuator: ur10e_pos_wrist_3_joint])
----------ur10e_flange
-----------ur10e_gripper_hand
------------ur10e_gripper_left_driver (Joints: ur10e_gripper_left_driver_joint-HINGE)
-------------ur10e_gripper_left_coupler
------------ur10e_gripper_left_spring_link (Joints: ur10e_gripper_left_spring_link_joint-HINGE)
-------------ur10e_gripper_left_follower (Joints: ur10e_

### Add cameras

Load camera specification

In [14]:
object_spec = mujoco.MjSpec.from_file(MODEL_PATH + "realsense_d435i.xml")
object_spec.copy_during_attach = True
object_spec.body("cam").mocap=0
print_body_tree(object_spec.worldbody, object_spec)

Body Tree for "world"
-cam


Mount camera to stands

In [15]:
cam_pos = scene.site("camera1_mount").pos
# cam_quat = map_pose(R=rot_y(np.pi / 2, out="R") @ rot_z(np.pi / 2, out="R"), out="Q")

attachment_frame = scene.body("camera1_stand").add_frame(pos=cam_pos)
attachment_frame.attach_body(object_spec.body("cam"), "RS1_")

In [16]:
cam_pos = scene.site("camera2_mount").pos
# cam_quat = map_pose(R=rot_z(-np.pi / 2, out="R"), out="Q")

attachment_frame = scene.body("camera2_stand").add_frame(pos=cam_pos)
attachment_frame.attach_body(object_spec.body("cam"), "RS2_")

### Prepare MJCF XML file

Redefine MJCF keys. Delete all keys and define Key 0 as the initial configuration

In [17]:
scene.keys

[]

In [18]:
_tmp = scene.to_xml()
while len(scene.keys)>1:
    scene.delete(scene.keys[-1])
for k in scene.keys:
    k.ctrl = np.concatenate([robot_ctrl, gripper_ctrl, np.zeros(len(scene.actuators) - len(robot_ctrl) - len(gripper_ctrl))])

Save MJCF scene to XML

In [19]:
scene.modelname = f"inspection_scene"
scene.option.timestep = 0.0002
scene.option.integrator = 3  # 0: Euler, 1: RK4, 2: implicit, 3: implicit_fast
with open(MODEL_PATH + "inspection_scene.xml", "w") as f:
    f.write(scene.to_xml())